# Error Simulation
Simulates a student error and compares how often the LLM performs it correctly

In [ ]:
import os
import json
from tqdm import tqdm
from collections import defaultdict

from openai import OpenAI
import pandas as pd

from src.datasets import ADataset, get_or_create_dataset
from src.equality import EqualityChecker, SemanticEqualityChecker
from src.evaluation import partial_match
from src.models.simulation import SimulateModel
from src.models.impl.groundtruth import GroundTruthMisconceptionModel
from src.models.impl.naive import DeepseekNaiveSimulateModel
from src.models.misconception import MisconceptionModel
from src.model_configurations import gpt_4_1_mini_det_config, deepseek_reasoner

In [ ]:
equivalence_check_path = "cache/semantic_equivalence_checker.pkl"

In [ ]:
eedi_dataset = get_or_create_dataset("eedi_data", n_limit=500)
print(f"We have {len(eedi_dataset)} EEDI questions")

datasets_by_datafolder = {
    "eedi_data": eedi_dataset
}

equality_model_config = gpt_4_1_mini_det_config
equality_client = OpenAI(base_url=equality_model_config["base_url"], api_key=os.environ.get(equality_model_config["api_key_var"], None))

semantic_equality_checker = SemanticEqualityChecker.load(equality_client, equivalence_check_path)

### Run Experiment

In [ ]:
save_every_n_questions = 10
n_retry_limit = 10

In [ ]:
def run_simulate(data_folder: str, dataset: ADataset, setting_name: str, m_misconception: MisconceptionModel, m_simulate: SimulateModel):
    results_folder = os.path.join(data_folder, "sim_results")
    os.makedirs(results_folder, exist_ok=True)
    response_file = os.path.join(results_folder, f"{setting_name}_responses_by_datapointid.json")

    responses_by_datapointid = defaultdict(dict)
    if os.path.exists(response_file): 
        with open(response_file, "r+") as f:
            responses_by_datapointid = json.load(f)

    for i in tqdm(list(range(len(dataset)))):
        datapoint_id = str(i)
        
        if datapoint_id in responses_by_datapointid: continue

        # save checkpoints
        if i % save_every_n_questions == 0: 
            with open(response_file, "w+") as f:
                json.dump(responses_by_datapointid, f)

        context = dataset[i]

        misconceptions,misconceptions_meta = m_misconception.propose(context, tried_misconceptions=[], num_misconceptions=1)
        misconception = misconceptions[0]
        misconception_id = misconceptions_meta["ids"][0]
        _, answer_meta = m_simulate.simulate(context, misconception)
        answer_meta["misconception_id"] = misconception_id
        answer_meta["misconception"] = misconception

        responses_by_datapointid[datapoint_id] = answer_meta

    # final save
    with open(response_file, "w+") as f:
        json.dump(responses_by_datapointid, f)

In [ ]:
run_simulate("eedi_data", eedi_dataset, f"deepseek-naive-deepseek-reasoner",
            m_misconception=GroundTruthMisconceptionModel(),
            m_simulate=DeepseekNaiveSimulateModel(deepseek_reasoner))

### Evaluation

In [ ]:
def analyze_responses(dataset: ADataset, responses_by_datapointid: dict, results_by_datapointid: dict, equality_checker: EqualityChecker):
    problem_by_datapointid = {
        str(dpid): dataset[dpid]["Problem"]
        for dpid in range(len(dataset))
    }
    gt_distractors_by_datapointid = { 
        str(dpid): dataset[dpid]["Choices"]["Distractors"]
        for dpid in range(len(dataset))
    }
    gt_correct_by_datapointid = { 
        str(dpid): dataset[dpid]["Choices"]["CorrectAnswer"] 
        for dpid in range(len(dataset))
    }
    num_cor_sol_steps_by_datapointid = {
        str(dpid): dataset[dpid]["Problem"]["NumReasoningSteps"]
        for dpid in range(len(dataset))
    }

    for dpid in tqdm(responses_by_datapointid.keys()):
        dpid = str(dpid)
        if dpid in results_by_datapointid: continue

        responses_of_datapointid = responses_by_datapointid[dpid]
        problem = problem_by_datapointid[dpid]
        question = problem["Question"]
        correct_answer = gt_correct_by_datapointid[dpid]
        misconception = responses_of_datapointid["misconception"]
        misconception_id = responses_of_datapointid["misconception_id"]
        misconception_answer = problem[misconception_id.replace("_Error", "")]
        same_misconception_ids = [k for k,v in problem.items() if k.endswith("_Error") and v.lower().strip() == misconception.lower().strip()]
        same_misconception_answers = [problem[mid.replace("_Error", "")] for mid in same_misconception_ids]
        groundtruth_distractors = gt_distractors_by_datapointid[dpid]
        llm_answer = responses_of_datapointid["answer"]

        not_deterministic = len(same_misconception_ids) > 1

        results_by_datapointid[dpid] = {
            "matches_chosen_misconception": equality_checker.is_equal(question, llm_answer, misconception_answer),
            "matches_same_misconception": partial_match(equality_checker, question, [llm_answer], same_misconception_answers),
            "matches_any_distractor": partial_match(equality_checker, question, [llm_answer], groundtruth_distractors),
            "not_deterministic": not_deterministic,
            "is_correct": equality_checker.is_equal(question, correct_answer, llm_answer),
            "problem_len_chars": len(question),
            "num_cor_sol_steps_by_datapointid": num_cor_sol_steps_by_datapointid[dpid]
        }

    return pd.DataFrame([{"Id": dpid, **result} for dpid,result in results_by_datapointid.items()])    

In [ ]:
for data_folder,dataset in datasets_by_datafolder.items():
    results_folder = os.path.join(data_folder, "sim_results")
    
    for filename in os.listdir(results_folder):
        if not filename.endswith("_responses_by_datapointid.json"): continue
        
        print(f"Processing responses: {os.path.join(results_folder, filename)}")
        with open(os.path.join(results_folder, filename), "r+") as f:
            responses_by_datapointid = json.loads(f.read())

            results_file = os.path.join(results_folder, f"{filename.replace('_responses_by_datapointid.json', '')}_results.csv")
            
            # load existing results
            results_by_datapointid = {}
            if os.path.exists(results_file):
                results_df = pd.read_csv(results_file)
                results_by_datapointid = {
                    str(row["Id"]): {k: row[k] for k in results_df.columns if k != "Id"}
                    for _, row in results_df.iterrows()
                }

            results_df = analyze_responses(dataset, responses_by_datapointid, results_by_datapointid, semantic_equality_checker)
            results_df.to_csv(results_file, index=False)

### Accuracy and Consistency

In [ ]:
import pandas as pd
from src.evaluation import get_mean_and_ci
df = pd.read_csv("eedi_data/sim_results/annotated/naive-deepseek-reasoner.csv")

In [ ]:
# how often does the LLMs answer match any of eedis distractors for that error
get_mean_and_ci(df["match_annot"])

In [ ]:
# how often does the LLMs answer match any value that can result from that error
get_mean_and_ci(df["valid"])

In [ ]:
# how often does the LLMs answer match the correct answer
get_mean_and_ci(df["matches_correct_annot"])